# 12-3. 계절성 포함 모델 구현 검증 (d=1, D=0 기존과 비교)

기존 모델(AR단독, AR+GARCH, AR+SV)을 d, D(계절 차분), 절편 규칙을 인자로 받는 식으로 일반화했다. d=1, D=0에서 기존 코드, 11시리즈 저장 결과(`11-1`)와 CRPS, MSE, 커버리지가 같은지 확인한다.

- **평가**: X3_l1, 채점 origin 22개(origin 35 (~2019Q3) ~ 56 (~2024Q4)), 자체 구간

## 1. 추가된 부분

**기존(d=1, 계절성 없음)**: `w_t = y_t - y_(t-1)`, `w_t = φ_1 w_(t-1) + ... + φ_p w_(t-p) + ε_t`(절편 없음). 분산은 AR단독 일정, GARCH(1,1), SV(1)이고 p는 AR단독 auto_arima(AICc)가 정한다.

| # | 추가된 부분 | 새 식 | D=0, P=0, d=1이면 기존과 같은 이유 |
|---|---|---|---|
| 1 | 계절 차분 D | `w_t = (1-B)^d (1-B^m)^D y_t`, m=4 | D=0이면 일반 차분만 남음 |
| 2 | 절편 규칙 | `d + D == 0`일 때만 절편 | d=1이면 절편 없음 |
| 3 | 계절 AR 항 | 시차 4, 8 추가, (p, P)는 `seasonal=True, m=4, max_Q=0`인 auto_arima가 선택 | P=0이면 일반 AR(p) |
| 4 | σ 전파 일반화 | AR 다항식 x 차분 다항식으로 충격반응 계산 | d번 누적합과 같음 |
| 5 | 역차분, 평균식 일반화 | 시차 목록과 절편 옵션을 받음 | 시차 1..p, 절편 없음이면 기존과 같음 |

바뀌지 않은 것: GARCH(1,1), SV(1), phi 상한 0.97, 90% 구간, H=4, 적합 실패 시 AR단독 대체.

## 2. 새 구현

위 표의 변경을 반영한 일반형이다.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from dataclasses import dataclass

import numpy as np
import pmdarima as pm
from arch.univariate import ARX, ConstantMean, GARCH, ZeroMean
from scipy import stats
from scipy.optimize import minimize

H, ALPHA, P_MAX = 4, 0.10, 3
Z = stats.norm.ppf(1 - ALPHA / 2)
XI_MEAN, XI_VAR, MAX_PHI = -1.2704, np.pi ** 2 / 2, 0.97


@dataclass(frozen=True)
class Spec:
    d: int = 1
    D: int = 0
    m: int = 4
    seasonal_ar: bool = False        # True면 계절 AR 차수 P도 AR단독 auto_arima가 탐색(D > 0이면 항상 탐색)

    @property
    def const(self):
        return self.d + self.D == 0


# ---------------------------------------------------------------- 차분·역차분·σ 전파
def diff_poly(sp):
    c = np.array([1.0])
    for _ in range(sp.d):
        c = np.convolve(c, [1.0, -1.0])
    for _ in range(sp.D):
        c = np.convolve(c, np.r_[1.0, np.zeros(sp.m - 1), -1.0])
    return c


def difference(y, sp):
    c, y = diff_poly(sp), np.asarray(y, float)
    k = len(c) - 1
    return y.copy() if k == 0 else np.convolve(y, c)[k:len(y)]


def undiff(w_fc, y, sp):
    c = diff_poly(sp)
    hist, out = list(map(float, y)), []
    for wf in w_fc:
        yt = wf - sum(c[i] * hist[-i] for i in range(1, len(c)))
        hist.append(yt)
        out.append(yt)
    return np.array(out)


def level_sigma(lags, ar, sp, innov_var):
    """h단계 수준 예측오차의 표준편차. 평균식 AR 다항식과 차분 다항식을 곱한 다항식으로 충격반응(psi)을 구한다."""
    ar_poly = np.zeros(max(lags) + 1 if len(lags) else 1)
    ar_poly[0] = 1.0
    for l, a in zip(lags, ar):
        ar_poly[l] -= a
    a_full = -np.convolve(ar_poly, diff_poly(sp))[1:]
    psi = np.zeros(H)
    for j in range(H):
        psi[j] = 1.0 if j == 0 else sum(a_full[k - 1] * psi[j - k] for k in range(1, min(j, len(a_full)) + 1))
    return np.sqrt(np.array([sum(psi[h - i] ** 2 * innov_var[i] for i in range(h + 1)) for h in range(H)]))


def lag_list(p, P, m):
    return sorted(set(range(1, p + 1)) | {m * j for j in range(1, P + 1)})


# ---------------------------------------------------------------- 구조 선택 (AR단독 auto_arima)
def select_structure(train, sp):
    kw = dict(d=sp.d, start_q=0, max_q=0, max_p=P_MAX, information_criterion="aicc", with_intercept=sp.const,
              suppress_warnings=True, error_action="ignore")
    if sp.D or sp.seasonal_ar:
        m = pm.auto_arima(train, seasonal=True, m=sp.m, D=sp.D, start_Q=0, max_Q=0, **kw)
        return m.order[0], m.seasonal_order[0], m
    m = pm.auto_arima(train, seasonal=False, **kw)
    return m.order[0], 0, m


def ar_forecast(m):
    fc, conf = m.predict(n_periods=H, return_conf_int=True, alpha=ALPHA)
    return np.asarray(fc), (conf[:, 1] - conf[:, 0]) / (2 * Z)


# ---------------------------------------------------------------- GARCH 동시
def garch_forecast(train, sp, lags):
    w = difference(train, sp)
    if lags:
        am = ARX(w, lags=lags, constant=sp.const, rescale=False)
    else:
        am = ConstantMean(w, rescale=False) if sp.const else ZeroMean(w, rescale=False)
    am.volatility = GARCH(p=1, q=1)
    res = am.fit(disp="off")
    f = res.forecast(horizon=H, reindex=False)
    ar = [res.params[f"y[{l}]"] for l in lags]
    fc = undiff(f.mean.values[-1], train, sp)
    sig = level_sigma(lags, ar, sp, f.residual_variance.values[-1])
    if not (np.all(np.isfinite(fc)) and np.all(np.isfinite(sig))):
        raise ValueError("non-finite forecast")
    return fc, sig


# ---------------------------------------------------------------- SV 동시 (QML + 칼만필터)
def is_ar_stable(lags, ar):
    if not len(lags):
        return True
    mx = max(lags)
    a = np.zeros(mx)
    for l, c in zip(lags, ar):
        a[l - 1] = c
    comp = np.zeros((mx, mx))
    comp[0, :] = a
    if mx > 1:
        comp[1:, :-1] = np.eye(mx - 1)
    return np.max(np.abs(np.linalg.eigvals(comp))) < 0.999


def sv_residuals(w, lags, ar, c):
    mx = max(lags) if len(lags) else 0
    return np.array([w[t] - c - sum(a * w[t - l] for a, l in zip(ar, lags)) for t in range(mx, len(w))])


def sv_qml_negloglik(z_centered, phi, sigma_eta2):
    P0 = sigma_eta2 / (1 - phi ** 2) if abs(phi) < 0.999 else sigma_eta2 * 100
    h_pred, P_pred, loglik = 0.0, P0, 0.0
    for z in z_centered:
        F = P_pred + XI_VAR
        v = z - h_pred
        loglik += -0.5 * (np.log(2 * np.pi * F) + v ** 2 / F)
        K = P_pred / F
        h_filt, P_filt = h_pred + K * v, P_pred * (1 - K)
        h_pred, P_pred = phi * h_filt, phi ** 2 * P_filt + sigma_eta2
    return -loglik


def _unpack(params, k, const):
    ar = params[:k]
    c = params[k] if const else 0.0
    j = k + (1 if const else 0)
    return ar, c, MAX_PHI * np.tanh(params[j]), np.exp(params[j + 1]) ** 2


def joint_sv_negloglik(params, w, lags, const):
    ar, c, phi, s2 = _unpack(params, len(lags), const)
    if not is_ar_stable(lags, ar):
        return 1e10
    z = np.log(sv_residuals(w, lags, ar, c) ** 2 + 1e-8)
    return sv_qml_negloglik(z - z.mean(), phi, s2)


def sv_forecast(train, sp, lags):
    w = difference(train, sp)
    k = len(lags)
    x0 = [0.0] * k + ([float(np.mean(w))] if sp.const else []) + [1.5, np.log(0.3)]
    res = minimize(joint_sv_negloglik, x0, args=(w, lags, sp.const), method="Nelder-Mead", options={"maxiter": 3000})
    ar, c, phi, s2 = _unpack(res.x, k, sp.const)
    if not is_ar_stable(lags, ar) or res.fun >= 1e9:
        raise ValueError("SV fit failed")
    z = np.log(sv_residuals(w, lags, ar, c) ** 2 + 1e-8)
    zm = z.mean()
    P, h = s2 / (1 - phi ** 2), 0.0
    for x in z - zm:
        K = P / (P + XI_VAR)
        hf, Pf = h + K * (x - h), P * (1 - K)
        h, P = phi * hf, phi ** 2 * Pf + s2
    past = list(w[-max(lags):]) if k else []
    mean_w = []
    for _ in range(H):
        pred = c + sum(a * past[-l] for a, l in zip(ar, lags))
        mean_w.append(pred)
        past.append(pred)
    v = [np.exp(zm - XI_MEAN + phi ** j * hf + (phi ** (2 * j) * Pf + s2 * sum(phi ** (2 * i) for i in range(j))) / 2) for j in range(1, H + 1)]
    fc, sig = undiff(mean_w, train, sp), level_sigma(lags, ar, sp, v)
    if not (np.all(np.isfinite(fc)) and np.all(np.isfinite(sig))):
        raise ValueError("non-finite forecast")
    return fc, sig


# ---------------------------------------------------------------- 공용 진입점
def forecast(model, train, sp=Spec()):
    """model: 'AR단독' | 'GARCH' | 'SV'. 반환: (예측 4개, σ 4개, 사용한 모델, (p, P)). 적합 실패 시 AR단독으로 대체한다."""
    p, P, m = select_structure(train, sp)
    if model == "AR단독":
        fc, sig = ar_forecast(m)
        return fc, sig, "AR단독", (p, P)
    lags = lag_list(p, P, sp.m)
    try:
        fc, sig = (garch_forecast if model == "GARCH" else sv_forecast)(train, sp, lags)
        return fc, sig, model, (p, P)
    except Exception:
        fc, sig = ar_forecast(m)
        return fc, sig, "AR대체", (p, P)


## 3. 기존 구현

`11-1`이 호출하는 점예측·σ 코드와 같다(d=1, 절편 없음, 계절성 없음). 이름 끝에 `_old`를 붙였다.

In [2]:
# 기존 구현: 11시리즈(11-1)가 호출하는 점예측·σ 코드와 같다(9시리즈와도 같음). d=1, 절편 없음 고정, 계절성 없음. 이름에 _old를 붙였다.
def difference_old(values, d):
    x = np.asarray(values, dtype=float)
    hist = []
    for _ in range(d):
        hist.append(x.copy())
        x = np.diff(x)
    return x, hist


def undiff_mean_old(diff_fc, hist, d):
    fc = np.asarray(diff_fc, dtype=float)
    for i in reversed(range(d)):
        fc = hist[i][-1] + np.cumsum(fc)
    return fc


def level_sigma_old(ar_coefs, d, innov_var):
    p = len(ar_coefs)
    psi = np.zeros(H)
    for j in range(H):
        psi[j] = 1.0 if j == 0 else sum(ar_coefs[k] * psi[j - 1 - k] for k in range(min(p, j)))
    for _ in range(d):
        psi = np.cumsum(psi)
    return np.sqrt(np.array([sum(psi[h - i] ** 2 * innov_var[i] for i in range(h + 1)) for h in range(H)]))


def select_p_old(train):
    m = pm.auto_arima(train, d=1, seasonal=False, start_q=0, max_q=0, max_p=P_MAX, information_criterion="aicc",
                      with_intercept=False, suppress_warnings=True, error_action="ignore")
    return m.order[0], m


def ar_fallback_old(m):
    fc, conf = m.predict(n_periods=H, return_conf_int=True, alpha=ALPHA)
    return np.asarray(fc), (conf[:, 1] - conf[:, 0]) / (2 * Z)


def garch_forecast_old(train, p):
    diff_train, hist = difference_old(train, 1)
    am = ZeroMean(diff_train, rescale=False) if p == 0 else ARX(diff_train, lags=p, constant=False, rescale=False)
    am.volatility = GARCH(p=1, q=1)
    res = am.fit(disp="off")
    f = res.forecast(horizon=H, reindex=False)
    ar = [res.params[f"y[{k}]"] for k in range(1, p + 1)]
    fc = undiff_mean_old(f.mean.values[-1], hist, 1)
    sig = level_sigma_old(ar, 1, f.residual_variance.values[-1])
    if not (np.all(np.isfinite(fc)) and np.all(np.isfinite(sig))):
        raise ValueError("non-finite forecast")
    return fc, sig


def is_ar_stable_old(ar_coefs):
    p = len(ar_coefs)
    if p == 0:
        return True
    companion = np.zeros((p, p))
    companion[0, :] = ar_coefs
    if p > 1:
        companion[1:, :-1] = np.eye(p - 1)
    return np.max(np.abs(np.linalg.eigvals(companion))) < 0.999


def sv_residuals_old(y, ar):
    p = len(ar)
    return np.array([y[t] - sum(ar[i] * y[t - 1 - i] for i in range(p)) for t in range(p, len(y))])


def joint_sv_negloglik_old(params, y, p):
    ar, phi, s2 = params[:p], MAX_PHI * np.tanh(params[p]), np.exp(params[p + 1]) ** 2
    if not is_ar_stable_old(ar):
        return 1e10
    z = np.log(sv_residuals_old(y, ar) ** 2 + 1e-8)
    return sv_qml_negloglik(z - z.mean(), phi, s2)       # 칼만필터 우도 함수는 새 구현과 같은 코드


def sv_forecast_old(train, p):
    y, hist = difference_old(train, 1)
    res = minimize(joint_sv_negloglik_old, [0.0] * p + [1.5, np.log(0.3)], args=(y, p), method="Nelder-Mead", options={"maxiter": 3000})
    if not is_ar_stable_old(res.x[:p]) or res.fun >= 1e9:
        raise ValueError("SV fit failed")
    ar, phi, s2 = res.x[:p], MAX_PHI * np.tanh(res.x[p]), np.exp(res.x[p + 1]) ** 2
    z = np.log(sv_residuals_old(y, ar) ** 2 + 1e-8)
    zm = z.mean()
    P, h = s2 / (1 - phi ** 2), 0.0
    for x in z - zm:
        K = P / (P + XI_VAR)
        hf, Pf = h + K * (x - h), P * (1 - K)
        h, P = phi * hf, phi ** 2 * Pf + s2
    past = list(y[-p:]) if p else []
    mean_diff = []
    for _ in range(H):
        pred = sum(ar[i] * past[-1 - i] for i in range(p))
        mean_diff.append(pred)
        past.append(pred)
    v = [np.exp(zm - XI_MEAN + phi ** k * hf + (phi ** (2 * k) * Pf + s2 * sum(phi ** (2 * j) for j in range(k))) / 2) for k in range(1, H + 1)]
    fc, sig = undiff_mean_old(mean_diff, hist, 1), level_sigma_old(ar, 1, v)
    if not (np.all(np.isfinite(fc)) and np.all(np.isfinite(sig))):
        raise ValueError("non-finite forecast")
    return fc, sig


def forecast_old(model, train):
    p, m = select_p_old(train)
    if model == "AR단독":
        return (*ar_fallback_old(m), "AR단독", p)
    try:
        fc, sig = (garch_forecast_old if model == "GARCH" else sv_forecast_old)(train, p)
        return fc, sig, model, p
    except Exception:
        fc, sig = ar_fallback_old(m)
        return fc, sig, "AR대체", p


## 4. CRPS, MSE, 커버리지 비교 (X3_l1, d=1, D=0)

In [3]:
import pandas as pd
import properscoring as ps
raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", "X3_l1"]].dropna()["X3_l1"].values.astype(float)
VS = vals.std()
U = stats.norm.ppf((np.arange(500) + 0.5) / 500)
SC = list(range(35, len(vals) - H + 1))                         # 채점 origin 35 (~2019Q3) ~ 56 (~2024Q4)
c11 = pd.read_pickle("../cache/cache_107_rank_compare_df.pkl")   # 11-1이 저장한 결과
c11 = c11[(c11["스케일"] == "원") & (c11["구간"] == "자체") & (c11["모델"].isin(["AR단독", "GARCH", "SV"]))]


def metrics(fc, sig, actual):
    crps = ps.crps_ensemble(actual, fc[:, None] + sig[:, None] * U[None, :]) / VS
    return ((actual - fc) / VS) ** 2, (np.abs(actual - fc) <= Z * sig) * 100.0, crps


acc = {}
for k in ("AR단독", "GARCH", "SV"):
    for tag in ("기존", "새"):
        acc[(k, tag)] = [[], [], []]
    for o in SC:
        tr, a = vals[:o], vals[o:o + H]
        for tag, (fc, sig) in (("기존", forecast_old(k, tr)[:2]), ("새", forecast(k, tr, Spec(d=1, D=0))[:2])):
            for lst, v in zip(acc[(k, tag)], metrics(fc, sig, a)):
                lst.extend(v)
rows = []
for k in ("AR단독", "GARCH", "SV"):
    g11 = c11[c11["모델"] == k]
    r = {"모델": k}
    for j, nm in enumerate(["CRPS", "MSE", "커버리지(%)"]):
        jj = {"CRPS": 2, "MSE": 0, "커버리지(%)": 1}[nm]
        r[f"{nm} 기존"] = np.mean(acc[(k, "기존")][jj])
        r[f"{nm} 새"] = np.mean(acc[(k, "새")][jj])
        r[f"{nm} 11시리즈"] = g11[{"MSE": "MSE", "커버리지(%)": "적중", "CRPS": "crps_ensemble"}[nm]].mean()
    rows.append(r)
M = pd.DataFrame(rows).set_index("모델")
display(M.round(4))

,CRPS 기존,CRPS 새,CRPS 11시리즈,MSE 기존,MSE 새,MSE 11시리즈,커버리지(%) 기존,커버리지(%) 새,커버리지(%) 11시리즈
모델,,,,,,,,,
AR단독,0.5931,0.5931,0.5931,1.1274,1.1274,1.1274,65.9091,65.9091,65.9091
GARCH,0.5795,0.5795,0.5795,1.0674,1.0674,1.0674,75.0000,75.0000,75.0000
SV,0.5783,0.5783,0.5783,1.0526,1.0526,1.0526,64.7727,64.7727,64.7727


## 5. 결론

**d=1, D=0에서 새 구현은 기존 코드, 11시리즈 저장 결과와 CRPS, MSE, 커버리지가 모두 같다.**

| 모델 | CRPS | MSE | 커버리지(%) |
|---|---|---|---|
| AR단독 | 0.5931 | 1.1274 | 65.9 |
| GARCH | 0.5795 | 1.0674 | 75.0 |
| SV | 0.5783 | 1.0526 | 64.8 |

- **확인한 범위**: X3_l1, d=1, D=0. 계절 항(D>0, P>0)과 d=0(절편)은 비교 대상이 없어 확인하지 않았다.
- **한계**: 계절 항이 들어가면 AR단독은 pmdarima의 SARIMA(교차항 포함), GARCH와 SV는 교차항 없는 계절 AR이라 구조가 약간 다르다.